# 06. 온톨로지와 RDF 지식그래프

**목적:** 03·08 노트북이 만든 로컬 자료를 작은 RDF 그래프로 표현하고, SPARQL 질의 5개가 pandas 집계와 일치하는지 확인합니다. 종로구·마포구, 공개 상권자료 외식업 10종 전체, 2023~2025년 **분기** 매출이 연구 범위입니다. 연구 상권에서 500m 이내인 다른 자치구 시설도 거리 집계의 일관성을 위해 포함합니다.

**RDF와 온톨로지는 다릅니다.** RDF는 `(주어, 관계, 목적어)`로 사실을 표현하는 데이터 모델입니다. 온톨로지는 어떤 개체·관계를 사용하며 그 뜻이 무엇인지 정하는 개념 체계입니다. 여기서는 RDFS 클래스·속성과 동일 단위·집계규칙 정의를 담은 작은 어휘를 `ontology.ttl`에, 실제 관측과 관계를 `knowledge_graph.ttl`에 저장합니다. Turtle은 이 그래프를 적는 파일 형식이고, SPARQL은 질의 언어입니다. 자동 추론이나 인과관계 발견을 구현하는 노트북은 아닙니다.

**실행 조건:** 03·08 완료 후 새 커널에서 위에서 아래로 실행합니다. `pandas`, `rdflib`가 필요하며 설치·API·네트워크 요청은 실행 셀에 없습니다. 모든 입력은 읽기 전용으로 사용합니다.

In [1]:
from pathlib import Path
from decimal import Decimal
import hashlib
import json
import platform
import importlib.metadata
import pandas as pd
from IPython.display import display, Markdown
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, DCTERMS

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/analysis/food_industry_panel.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 폴더에서 03·08 노트북을 먼저 실행하세요.')
OUT = ROOT / 'data/analysis/knowledge_graph'
REPORTS = ROOT / 'reports'
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
DISTRICTS = ['종로구', '마포구']
EXPECTED_INDUSTRY_IDS = [f'CS100{i:03d}' for i in range(1, 11)]
QUARTERS = [f'{year}{q}' for year in range(2023, 2026) for q in range(1, 5)]
KEY = ['area_id', 'industry_id', 'quarter']
print({'Python': platform.python_version(), 'rdflib': importlib.metadata.version('rdflib')})

{'Python': '3.12.14', 'rdflib': '7.6.0'}


## 1. 직접 입력 파일과 연구 범위

ID는 문자열로 읽고 금액은 원본 CSV의 숫자 문자열을 `Decimal`로 변환합니다. 미제공 매출은 금액 리터럴을 만들지 않습니다. 제공 여부와 값의 존재가 일치하지 않으면 실행을 중단합니다. 아래 출처는 **직접 읽은 파일 수준**이며 원 API 응답의 개별 행까지 복원하는 계보는 아닙니다.

08의 고정 범위 메타데이터에서 패널·업종 목록 해시와 수집 실행 ID를 읽고, 시설 목록·상세도 같은 메타데이터에 고정된 경로와 해시로 검증한다. 수집 중 계속 바뀌는 live 포인터는 읽지 않는다. 출처 표의 행 수는 CSV 행 수이며 JSON 메타데이터는 문서 1개로 표시한다.

In [2]:
scope_metadata = json.loads((ROOT / 'reports/08_food_scope_metadata.json').read_text(encoding='utf-8'))
snapshot_sources = {row['role']: row for row in scope_metadata['exposure_snapshot_linkage']['source_files']}
INPUTS = {
    'panel': 'data/analysis/food_industry_panel.csv',
    'catalog': 'data/analysis/food_industry_catalog.csv',
    'scope_metadata': 'reports/08_food_scope_metadata.json',
    'areas': 'data/analysis/study_areas.csv',
    'performances': 'data/analysis/performance_facility_linkage.csv',
    'active': 'data/analysis/performance_facility_quarter.csv',
    'distances': 'data/analysis/venue_area_distances_800m.csv',
    'facilities': snapshot_sources['facilities_list']['path'],
    'details': snapshot_sources['facility_details']['path'],
}
frames, source_rows = {}, []
for name, relative in INPUTS.items():
    source = ROOT / relative
    if not source.exists():
        raise FileNotFoundError(f'필요한 로컬 파일이 없습니다: {relative}')
    if source.suffix == '.json':
        scope_metadata = json.loads(source.read_text(encoding='utf-8'))
        source_count = 1  # JSON 문서 1개; CSV 행 수와 구분한다.
    else:
        frames[name] = pd.read_csv(source, dtype=str)
        source_count = len(frames[name])
    source_rows.append({'source': name, 'path': relative, 'rows': source_count,
                        'sha256': hashlib.sha256(source.read_bytes()).hexdigest()})
source_manifest = pd.DataFrame(source_rows)
for name, role in [('facilities', 'facilities_list'), ('details', 'facility_details')]:
    assert source_manifest.set_index('source').loc[name, 'sha256'] == snapshot_sources[role]['sha256'], f'08과 시설 입력 버전이 다릅니다: {role}'
assert len(frames['details']) == scope_metadata['exposure_snapshot_linkage']['facility_detail_rows']
display(source_manifest[['source', 'path', 'rows']])

,source,path,rows
0,panel,data/analysis/food_industry_panel.csv,18120
1,catalog,data/analysis/food_industry_catalog.csv,10
2,scope_metadata,reports/08_food_scope_metadata.json,1
3,areas,data/analysis/study_areas.csv,151
4,performances,data/analysis/performance_facility_linkage.csv,30047
5,active,data/analysis/performance_facility_quarter.csv,31988
6,distances,data/analysis/venue_area_distances_800m.csv,10493
7,facilities,data/processed/kopis/facilities_list_seoul.csv,1755
8,details,data/analysis/kopis_snapshots/20261004_092358_...,1755


In [3]:
def as_bool(series):
    assert series.isin(['True', 'False']).all(), '불리언 입력을 확인하세요.'
    return series.eq('True')

catalog = frames['catalog'].sort_values('industry_id').copy()
assert as_bool(catalog['included']).all()
assert catalog['industry_id'].tolist() == EXPECTED_INDUSTRY_IDS
assert not catalog[['industry_id', 'industry']].duplicated().any()
INDUSTRIES = catalog['industry_id'].tolist()
for name in ['panel', 'catalog']:
    assert scope_metadata[f'{name}_sha256'] == source_manifest.set_index('source').loc[name, 'sha256']
panel = frames['panel'].copy()
areas = frames['areas'].copy()
assert set(panel['district']) <= set(DISTRICTS)
assert set(panel['industry_id']) == set(INDUSTRIES)
assert set(map(tuple, panel[['industry_id', 'industry']].drop_duplicates().values)) == set(
    map(tuple, catalog[['industry_id', 'industry']].values))
assert set(panel['quarter']) == set(QUARTERS)
assert not panel.duplicated(KEY).any()
assert len(panel) == panel['area_id'].nunique() * len(INDUSTRIES) * len(QUARTERS)
assert not areas['area_id'].duplicated().any()
assert set(panel['area_id']) == set(areas['area_id'])
panel['provided'] = as_bool(panel['sales_record_present'])
assert panel['provided'].equals(panel['total_sales'].notna())
panel['sales_decimal'] = panel['total_sales'].map(
    lambda value: Decimal(value) if pd.notna(value) else None)
assert all(value.is_finite() for value in panel.loc[panel['provided'], 'sales_decimal'])

In [4]:
facilities = frames['facilities'].merge(
    frames['details'][['mt10id', 'lo', 'la']], on='mt10id', how='left', validate='one_to_one')
facilities['longitude'] = pd.to_numeric(facilities['lo'], errors='coerce')
facilities['latitude'] = pd.to_numeric(facilities['la'], errors='coerce')
facilities['coordinate_available'] = (facilities['longitude'].between(124, 132) &
                                     facilities['latitude'].between(33, 39))
distances = frames['distances'].copy()
distances['distance_value'] = distances['distance_m'].map(Decimal)
assert distances['distance_value'].ge(0).all()
near = distances.loc[distances['distance_value'].le(500)].copy()
assert not near.duplicated(['area_id', 'mt10id']).any()
assert set(near['area_id']) <= set(areas['area_id'])
facility_ids = (set(facilities.loc[facilities['gugunnm'].isin(DISTRICTS), 'mt10id']) |
                set(near['mt10id']))
facilities = facilities.loc[facilities['mt10id'].isin(facility_ids)].copy()
assert set(near['mt10id']) <= set(facilities.loc[facilities['coordinate_available'], 'mt10id'])

In [5]:
performances = frames['performances'].loc[
    frames['performances']['mt10id'].isin(facility_ids)].copy()
assert not performances['mt20id'].duplicated().any()
assert performances['link_method'].isin(
    ['official_detail', 'exact_unique_name', 'prefix_unique_name']).all()
performances['official'] = performances['link_method'].eq('official_detail')
performances['status'] = performances['official'].map({True: 'official', False: 'provisional'})
performances['coordinates'] = as_bool(performances['valid_coordinates'])
assert (performances.loc[performances['official'], 'mt10id'] ==
        performances.loc[performances['official'], 'official_id']).all()
active = frames['active'].loc[frames['active']['mt20id'].isin(performances['mt20id'])].copy()
assert not active.duplicated(['mt20id', 'quarter']).any()
assert set(active['quarter']) <= set(QUARTERS)
assert set(active['mt20id']) <= set(performances.loc[performances['coordinates'], 'mt20id'])
check_active = active.merge(performances[['mt20id', 'mt10id', 'link_method']],
                           on='mt20id', suffixes=('_active', '_link'), validate='many_to_one')
assert check_active['mt10id_active'].eq(check_active['mt10id_link']).all()
assert check_active['link_method_active'].eq(check_active['link_method_link']).all()

In [6]:
coverage = pd.DataFrame([
    ('상권', len(areas), len(frames['areas'])),
    ('매출 관측', len(panel), len(frames['panel'])),
    ('공연시설', len(facilities), len(frames['facilities'])),
    ('등록 공연 ID', len(performances), len(frames['performances'])),
    ('공연-분기 관계', len(active), len(frames['active'])),
    ('시설-상권 거리 관계', len(near), len(frames['distances'])),
], columns=['대상', '그래프 포함', '입력 파일 전체'])
display(catalog[['industry_id', 'industry', 'selection_order']])
display(coverage)
display(facilities.groupby('gugunnm').agg(
    시설수=('mt10id', 'size'), 좌표확보=('coordinate_available', 'sum')))
display(Markdown(
    f"매출 {len(panel):,}개 관측 중 제공 {int(panel['provided'].sum()):,}개, "
    f"미제공 {int((~panel['provided']).sum()):,}개입니다. "
    f"서울 공연 {len(frames['performances']):,}개 중 {len(performances):,}개를 포함합니다."))

,industry_id,industry,selection_order
0,CS100001,한식음식점,1
1,CS100002,중식음식점,2
2,CS100003,일식음식점,3
3,CS100004,양식음식점,4
4,CS100005,제과점,5
5,CS100006,패스트푸드점,6
6,CS100007,치킨전문점,7
7,CS100008,분식전문점,8
8,CS100009,호프-간이주점,9
9,CS100010,커피-음료,10


,대상,그래프 포함,입력 파일 전체
0,상권,151,151
1,매출 관측,18120,18120
2,공연시설,801,1755
3,등록 공연 ID,12839,30047
4,공연-분기 관계,14202,31988
5,시설-상권 거리 관계,5670,10493


,시설수,좌표확보
gugunnm,,
동대문구,4,4
마포구,296,296
서대문구,51,51
성북구,23,23
용산구,7,7
은평구,1,1
종로구,376,376
중구,43,43


매출 18,120개 관측 중 제공 7,661개, 미제공 10,459개입니다. 서울 공연 30,047개 중 12,839개를 포함합니다.

**해석:** 시설 범위는 두 연구 자치구의 시설 전체와 연구 상권의 500m 거리 관계에 등장하는 시설의 합집합입니다. 공연은 그 시설에 연결된 ID만 포함합니다. 따라서 서울 전체 공연 그래프는 아닙니다. 좌표가 없는 연구지역 시설·공연도 상태를 보존하지만, 거리·활성 분기 집계에는 사용할 수 없습니다. 입력 단계에서 시설조차 연결되지 않은 공연은 지역을 판별할 수 없어 이 부분 그래프에서 제외됩니다.

공연 활성 분기는 03에서 **공연기간과 분기의 겹침**으로 만든 관계를 그대로 읽습니다. 등록 공연 ID 수는 상연 횟수·관객수·일별 공연량이 아닙니다. 동일 공연이 여러 분기 또는 여러 상권에 나타날 수 있어 합계를 전체 고유 공연 수로 해석하면 안 됩니다.

## 2. 작은 어휘 정의

핵심 관계는 `매출관측 → 상권·업종·분기`, `공연 → 시설연결 → 시설`, `공연 → 활성분기`, `거리관계 → 시설·상권`입니다. 시설연결을 독립 개체로 만들어 **공식 ID 확인**과 **시설명 기반 잠정 연결**의 상태와 방법을 붙입니다. 시설명 일치는 공식 동일성으로 승격하지 않습니다.

RDFS의 domain/range는 의미를 설명하는 선언이며 데이터 검증 제약과 같지 않습니다. 이 노트북의 실제 검증은 아래 `assert`와 pandas 대조로 수행합니다. 어휘 URI는 로컬 식별용 `https://example.org/grad-study/`를 사용하며 접속하지 않습니다.

In [7]:
EX = Namespace('https://example.org/grad-study/')
vocab, graph = Graph(), Graph()
for target in [vocab, graph]:
    for prefix, namespace in [('ex', EX), ('rdf', RDF), ('rdfs', RDFS),
                              ('xsd', XSD), ('dct', DCTERMS)]:
        target.bind(prefix, namespace)

def node(kind, identifier):
    return EX[f'{kind}/{identifier}']

def add(target, subject, **properties):
    for predicate, value in properties.items():
        if value is not None and not pd.isna(value):
            target.add((subject, EX[predicate], value if isinstance(value, URIRef) else Literal(value)))

classes = {'StudyArea': '상권', 'Industry': '업종', 'Quarter': '분기',
           'Performance': '등록 공연', 'Facility': '공연시설', 'FacilityLink': '공연-시설 연결',
           'SalesObservation': '상권·업종·분기 매출관측', 'ProximityRelation': '시설-상권 거리관계',
           'SourceFile': '직접 입력 파일', 'Unit': '단위', 'Rule': '집계·거리 규칙',
           'LinkStatus': '연결 확인 상태', 'Dataset': '부분 지식그래프'}
for term, label in classes.items():
    vocab.add((EX[term], RDF.type, RDFS.Class))
    vocab.add((EX[term], RDFS.label, Literal(label, lang='ko')))

In [8]:
# 범용 메타데이터의 domain은 생략하여 부정확한 클래스 추론을 피합니다.
property_specs = [
    ('areaCode', '상권 코드', 'StudyArea', XSD.string),
    ('industryCode', '업종 코드', 'Industry', XSD.string),
    ('quarterCode', '연도+분기 코드', 'Quarter', XSD.string),
    ('districtName', '자치구 이름', None, XSD.string),
    ('inArea', '매출관측의 상권', 'SalesObservation', EX.StudyArea),
    ('inIndustry', '매출관측의 업종', 'SalesObservation', EX.Industry),
    ('inQuarter', '매출관측의 분기', 'SalesObservation', EX.Quarter),
    ('totalSales', '분기 총매출액', 'SalesObservation', XSD.decimal),
    ('salesProvided', '매출 원천 행 제공 여부', 'SalesObservation', XSD.boolean),
    ('salesUnit', '매출액 단위', 'SalesObservation', EX.Unit),
    ('hasFacilityLink', '공연의 시설 연결 주장', 'Performance', EX.FacilityLink),
    ('linkedPerformance', '연결 대상 공연', 'FacilityLink', EX.Performance),
    ('linkedFacility', '연결 대상 시설', 'FacilityLink', EX.Facility),
    ('linkMethod', '상위 전처리의 연결 방법', 'FacilityLink', XSD.string),
    ('linkStatus', '공식 또는 잠정 연결 상태', 'FacilityLink', EX.LinkStatus),
    ('nameOfficialConflict', '공식 ID와 시설명 후보의 충돌', 'FacilityLink', XSD.boolean),
    ('activeQuarter', '좌표·기간 확인 공연의 기간 겹침 분기', 'Performance', EX.Quarter),
    ('coordinateAvailable', '좌표 확인 여부', None, XSD.boolean),
    ('longitude', '경도 EPSG:4326', 'Facility', XSD.decimal),
    ('latitude', '위도 EPSG:4326', 'Facility', XSD.decimal),
    ('nearArea', '거리관계의 상권', 'ProximityRelation', EX.StudyArea),
    ('nearFacility', '거리관계의 시설', 'ProximityRelation', EX.Facility),
    ('distanceMetres', '현재 상권 폴리곤까지 최소거리', 'ProximityRelation', XSD.decimal),
    ('distanceUnit', '거리 단위', 'ProximityRelation', EX.Unit),
    ('rule', '적용 집계 또는 거리 규칙', None, EX.Rule),
    ('sourcePath', '저장소 기준 입력 파일 경로', 'SourceFile', XSD.string),
    ('sha256', '입력 파일 SHA-256', 'SourceFile', XSD.string),
    ('sourceRows', '입력 CSV 행 수 또는 JSON 문서 수', 'SourceFile', XSD.integer),
    ('officialIdentifier', '자료 제공처의 공식 개체 ID', None, XSD.string),
    ('startDate', '공연 시작일', 'Performance', XSD.date),
    ('endDate', '공연 종료일', 'Performance', XSD.date),
]
for term, label, domain, range_uri in property_specs:
    vocab.add((EX[term], RDF.type, RDF.Property))
    vocab.add((EX[term], RDFS.label, Literal(label, lang='ko')))
    vocab.add((EX[term], RDFS.range, range_uri))
    if domain:
        vocab.add((EX[term], RDFS.domain, EX[domain]))
display(pd.DataFrame(property_specs, columns=['속성', '뜻', '주어 클래스', '값 종류']).iloc[:17])

,속성,뜻,주어 클래스,값 종류
0,areaCode,상권 코드,StudyArea,http://www.w3.org/2001/XMLSchema#string
1,industryCode,업종 코드,Industry,http://www.w3.org/2001/XMLSchema#string
2,quarterCode,연도+분기 코드,Quarter,http://www.w3.org/2001/XMLSchema#string
3,districtName,자치구 이름,NaN,http://www.w3.org/2001/XMLSchema#string
4,inArea,매출관측의 상권,SalesObservation,https://example.org/grad-study/StudyArea
5,inIndustry,매출관측의 업종,SalesObservation,https://example.org/grad-study/Industry
6,inQuarter,매출관측의 분기,SalesObservation,https://example.org/grad-study/Quarter
7,totalSales,분기 총매출액,SalesObservation,http://www.w3.org/2001/XMLSchema#decimal
8,salesProvided,매출 원천 행 제공 여부,SalesObservation,http://www.w3.org/2001/XMLSchema#boolean
9,salesUnit,매출액 단위,SalesObservation,https://example.org/grad-study/Unit


In [9]:
for term, cls, label, description in [
    ('KRW', 'Unit', '원', '서울 상권 자료의 분기 총매출액 단위. 물가 조정 전 명목 금액.'),
    ('metre', 'Unit', '미터', '투영좌표계에서 계산한 공간 거리의 단위.'),
    ('official', 'LinkStatus', '공식 ID 확인', '공연 상세자료의 mt10id로 확인한 연결.'),
    ('provisional', 'LinkStatus', '시설명 잠정 연결', '시설명 정규화 일치 기반 후보. 공식 ID 확인과 구분.'),
    ('quarterlySales', 'Rule', '상권·업종·분기 총매출', '분기 값을 일별로 배분하지 않음. 미제공 금액 생략.'),
    ('quarterOverlap', 'Rule', '공연기간과 분기 겹침', '03 산출물 재사용. 좌표와 기간을 확인한 공연 ID의 분기 겹침.'),
    ('polygon500m', 'Rule', '현재 상권 폴리곤 500m 이내',
     '03 산출물 재사용. 현재 상권 폴리곤까지 최소거리 500m 이하; 내부는 0m. 보행거리나 중심점 거리가 아님.'),
]:
    graph.add((EX[term], RDF.type, EX[cls]))
    graph.add((EX[term], RDFS.label, Literal(label, lang='ko')))
    graph.add((EX[term], RDFS.comment, Literal(description, lang='ko')))
# 후속 도구가 작은 ontology 파일만 읽어 동일 단위·규칙 정의를 확인하게 합니다.
for term in ['KRW', 'metre', 'quarterlySales', 'quarterOverlap', 'polygon500m']:
    for triple in graph.triples((EX[term], None, None)):
        vocab.add(triple)

for row in source_manifest.itertuples(index=False):
    source = node('source', row.source)
    graph.add((source, RDF.type, EX.SourceFile))
    add(graph, source, sourcePath=row.path, sha256=row.sha256, sourceRows=int(row.rows))

dataset = EX.dataset

In [10]:
graph.add((dataset, RDF.type, EX.Dataset))
graph.add((dataset, RDFS.label, Literal('종로구·마포구 공연과 외식업 10종 부분 지식그래프', lang='ko')))
graph.add((dataset, RDFS.comment, Literal(
    f'2023~2025년 분기. 현재 상권 경계를 과거 분기에 공통 적용. 현재 좌표 미확인 시설 {int((~facilities["coordinate_available"]).sum())}개; 과거 시설 위치 미검증. 시설명 잠정 연결의 오연결 가능성이 있는 부분 그래프. '
    '매출값 미제공은 0원이 아님. 공연수는 등록 ID 수이며 상연 횟수·관객수가 아님.', lang='ko')))
for name in INPUTS:
    graph.add((dataset, DCTERMS.source, node('source', name)))
for rule in ['quarterlySales', 'quarterOverlap', 'polygon500m']:
    graph.add((dataset, EX.rule, EX[rule]))
display(pd.DataFrame([(classes[c], c) for c in classes], columns=['개념', '클래스']))

,개념,클래스
0,상권,StudyArea
1,업종,Industry
2,분기,Quarter
3,등록 공연,Performance
4,공연시설,Facility
5,공연-시설 연결,FacilityLink
6,상권·업종·분기 매출관측,SalesObservation
7,시설-상권 거리관계,ProximityRelation
8,직접 입력 파일,SourceFile
9,단위,Unit


**해석:** 공연 ID와 시설 ID 자체는 자료 제공처의 공식 식별자입니다. 그러나 두 ID 사이의 연결까지 공식 확인되었다는 뜻은 아닙니다. `FacilityLink.linkStatus`가 그 차이를 표현합니다. 거리관계가 없는 시설은 멀리 있다는 결론을 내릴 수 없습니다. 좌표가 없거나 부분 그래프의 수집 범위 밖일 수도 있습니다.

## 3. 개체와 관측을 RDF로 변환

상권·업종·분기의 조합마다 안정적인 URI를 하나 만들고, 금액은 제공된 관측에만 저장합니다. URI는 저장소에 국한한 식별자이며 API 주소가 아닙니다. 분기·거리 집계 규칙과 입력 파일을 관계에 함께 기록합니다.

In [11]:
for row in areas.itertuples(index=False):
    area = node('area', row.area_id)
    graph.add((area, RDF.type, EX.StudyArea))
    graph.add((area, RDFS.label, Literal(row.area_name, lang='ko')))
    add(graph, area, areaCode=row.area_id, districtName=row.district)
    graph.add((area, DCTERMS.source, node('source', 'areas')))
for row in catalog[['industry_id', 'industry']].itertuples(index=False):
    industry = node('industry', row.industry_id)
    graph.add((industry, RDF.type, EX.Industry))
    graph.add((industry, RDFS.label, Literal(row.industry, lang='ko')))
    add(graph, industry, industryCode=row.industry_id)
    graph.add((industry, DCTERMS.source, node('source', 'catalog')))
for quarter in QUARTERS:
    resource = node('quarter', quarter)
    graph.add((resource, RDF.type, EX.Quarter))
    graph.add((resource, RDFS.label, Literal(f'{quarter[:4]}년 {quarter[-1]}분기', lang='ko')))
    add(graph, resource, quarterCode=quarter)

In [12]:
for row in facilities.itertuples(index=False):
    facility = node('facility', row.mt10id)
    graph.add((facility, RDF.type, EX.Facility))
    graph.add((facility, RDFS.label, Literal(row.fcltynm, lang='ko')))
    add(graph, facility, officialIdentifier=row.mt10id, districtName=row.gugunnm,
        coordinateAvailable=bool(row.coordinate_available))
    graph.add((facility, DCTERMS.source, node('source', 'facilities')))
    graph.add((facility, DCTERMS.source, node('source', 'details')))
    if row.coordinate_available:
        add(graph, facility, longitude=Decimal(row.lo), latitude=Decimal(row.la))

for row in performances.itertuples(index=False):
    performance, link = node('performance', row.mt20id), node('link', row.mt20id)
    graph.add((performance, RDF.type, EX.Performance))
    graph.add((performance, RDFS.label, Literal(row.prfnm, lang='ko')))
    add(graph, performance, officialIdentifier=row.mt20id,
        coordinateAvailable=bool(row.coordinates), hasFacilityLink=link)
    if row.valid_period == 'True':
        graph.add((performance, EX.startDate, Literal(row.start_date, datatype=XSD.date)))
        graph.add((performance, EX.endDate, Literal(row.end_date, datatype=XSD.date)))
    graph.add((link, RDF.type, EX.FacilityLink))
    add(graph, link, linkedPerformance=performance, linkedFacility=node('facility', row.mt10id),
        linkMethod=row.link_method, linkStatus=EX[row.status],
        nameOfficialConflict=row.name_official_conflict == 'True')
    for resource in [performance, link]:
        graph.add((resource, DCTERMS.source, node('source', 'performances')))

In [13]:
for row in active.itertuples(index=False):
    performance = node('performance', row.mt20id)
    add(graph, performance, activeQuarter=node('quarter', row.quarter), rule=EX.quarterOverlap)
    graph.add((performance, DCTERMS.source, node('source', 'active')))
for row in near.itertuples(index=False):
    relation = node('distance', f'{row.area_id}-{row.mt10id}')
    graph.add((relation, RDF.type, EX.ProximityRelation))
    add(graph, relation, nearArea=node('area', row.area_id), nearFacility=node('facility', row.mt10id),
        distanceMetres=row.distance_value, distanceUnit=EX.metre, rule=EX.polygon500m)
    graph.add((relation, DCTERMS.source, node('source', 'distances')))
for row in panel.itertuples(index=False):
    observation = node('sales', f'{row.area_id}-{row.industry_id}-{row.quarter}')
    graph.add((observation, RDF.type, EX.SalesObservation))
    add(graph, observation, inArea=node('area', row.area_id),
        inIndustry=node('industry', row.industry_id), inQuarter=node('quarter', row.quarter),
        salesProvided=bool(row.provided), salesUnit=EX.KRW, rule=EX.quarterlySales)
    graph.add((observation, DCTERMS.source, node('source', 'panel')))
    if row.provided:
        add(graph, observation, totalSales=row.sales_decimal)

In [14]:
entity_counts = pd.DataFrame([
    {'class': cls, '개념': label, '개체수': len(set(graph.subjects(RDF.type, EX[cls])))}
    for cls, label in classes.items()])
display(entity_counts)
example = node('sales', '-'.join(panel.iloc[0][KEY]))
example_rows = [(str(p).split('/')[-1].split('#')[-1], str(o)) for p, o in graph.predicate_objects(example)]
display(pd.DataFrame(example_rows, columns=['관계', '목적어']))
print(f'데이터 트리플 {len(graph):,}개 / 어휘 트리플 {len(vocab):,}개')

,class,개념,개체수
0,StudyArea,상권,151
1,Industry,업종,10
2,Quarter,분기,12
3,Performance,등록 공연,12839
4,Facility,공연시설,801
5,FacilityLink,공연-시설 연결,12839
6,SalesObservation,상권·업종·분기 매출관측,18120
7,ProximityRelation,시설-상권 거리관계,5670
8,SourceFile,직접 입력 파일,9
9,Unit,단위,2


,관계,목적어
0,type,https://example.org/grad-study/SalesObservation
1,inArea,https://example.org/grad-study/area/3001494
2,inIndustry,https://example.org/grad-study/industry/CS100001
3,inQuarter,https://example.org/grad-study/quarter/20231
4,salesProvided,true
5,salesUnit,https://example.org/grad-study/KRW
6,rule,https://example.org/grad-study/quarterlySales
7,source,https://example.org/grad-study/source/panel
8,totalSales,46431279586.0


데이터 트리플 432,888개 / 어휘 트리플 161개


**해석:** 미제공 관측에도 상권·업종·분기와 `salesProvided=false`는 존재하지만 `totalSales`는 없습니다. RDF에서 금액이 없다는 사실만으로 매출이 0원이라고 결론 내릴 수 없습니다. 반대로 입력에 실제 0원 관측이 있다면 `salesProvided=true`와 금액 0을 함께 저장합니다.

## 4. SPARQL 질의 5개와 pandas 대조

질의는 생성한 로컬 그래프에서만 실행합니다. 질의문은 아래 코드 셀에만 유지하여 후속 분석 노트북에서 같은 질문을 재사용할 수 있도록 합니다. 문자열로 읽은 금액을 RDF와 pandas 양쪽에서 `Decimal`로 합산하므로 금액 비교에는 부동소수점 반올림을 사용하지 않습니다.

In [15]:
PREFIX = 'PREFIX ex: <https://example.org/grad-study/>\n'
queries, query_results, validations = {}, {}, []

def run_query(name, text):
    queries[name] = PREFIX + text.strip()
    result = graph.query(queries[name])
    table = pd.DataFrame([[v.toPython() if v is not None else None for v in row]
                          for row in result], columns=[str(v) for v in result.vars])
    query_results[name] = table
    return table

def compare(name, rdf, expected, keys):
    columns = list(expected.columns)
    left = rdf[columns].sort_values(keys).reset_index(drop=True)
    right = expected.sort_values(keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(left, right, check_dtype=False, check_exact=True)
    validations.append({'check': name, 'status': 'passed', 'rows_checked': len(right)})
    display(left.head(8))
    print(f'{name}: {len(right):,}행 pandas와 정확히 일치')

### 질의 1 — 분기·업종별 제공/미제공 관측 수

`totalSales`가 존재하는 횟수와 `salesProvided` 플래그를 함께 집계합니다. 매출 미제공을 0원으로 바꾸는 오류를 탐지하는 기본 질의입니다.

In [16]:
q1 = run_query('q1_sales_coverage', """
SELECT ?quarter ?industry_id (COUNT(?obs) AS ?observations)
       (SUM(IF(?provided, 1, 0)) AS ?provided_count) (COUNT(?sales) AS ?value_count)
WHERE {
  ?obs a ex:SalesObservation; ex:inQuarter/ex:quarterCode ?quarter;
       ex:inIndustry/ex:industryCode ?industry_id; ex:salesProvided ?provided.
  OPTIONAL { ?obs ex:totalSales ?sales. }
} GROUP BY ?quarter ?industry_id ORDER BY ?quarter ?industry_id
""")
expected1 = panel.groupby(['quarter', 'industry_id']).agg(
    observations=('area_id', 'size'), provided_count=('provided', 'sum'),
    value_count=('total_sales', 'count')).reset_index()
compare('q1_coverage', q1, expected1, ['quarter', 'industry_id'])

,quarter,industry_id,observations,provided_count,value_count
0,20231,CS100001,151,123,123
1,20231,CS100002,151,41,41
2,20231,CS100003,151,40,40
3,20231,CS100004,151,53,53
4,20231,CS100005,151,42,42
5,20231,CS100006,151,39,39
6,20231,CS100007,151,45,45
7,20231,CS100008,151,74,74


q1_coverage: 120행 pandas와 정확히 일치


**해석:** 관측 수와 금액값 수의 차이는 미제공 수입니다. 금액이 없는 관측도 분모에서 빠지지 않아 자료 제공률을 확인할 수 있습니다.

### 질의 2 — 제공된 분기 매출 합계

자치구·업종·분기별 제공 관측의 금액만 합칩니다. 이는 연구 상권의 제공분 합계이며 결측 상권을 포함한 자치구 전체 매출 추정치가 아닙니다. 상권이 중첩되는 경우 중복 합산 가능성도 별도 확인이 필요합니다.

In [17]:
q2 = run_query('q2_observed_sales_sum', """
SELECT ?district ?industry_id ?quarter (COUNT(?obs) AS ?observations)
       (SUM(?sales) AS ?observed_sales_krw)
WHERE {
  ?obs a ex:SalesObservation; ex:salesProvided true; ex:totalSales ?sales;
       ex:inArea/ex:districtName ?district;
       ex:inIndustry/ex:industryCode ?industry_id; ex:inQuarter/ex:quarterCode ?quarter.
} GROUP BY ?district ?industry_id ?quarter ORDER BY ?district ?industry_id ?quarter
""")
expected2 = panel.loc[panel['provided']].groupby(['district', 'industry_id', 'quarter']).agg(
    observations=('area_id', 'size'), observed_sales_krw=('sales_decimal', 'sum')).reset_index()
compare('q2_sales_decimal_sum', q2, expected2, ['district', 'industry_id', 'quarter'])

,district,industry_id,quarter,observations,observed_sales_krw
0,마포구,CS100001,20231,63,177950601126.0
1,마포구,CS100001,20232,64,177645965394.0
2,마포구,CS100001,20233,64,176157682773.0
3,마포구,CS100001,20234,64,198211105416.0
4,마포구,CS100001,20241,65,188788173914.0
5,마포구,CS100001,20242,65,190643574715.0
6,마포구,CS100001,20243,65,179997046997.0
7,마포구,CS100001,20244,65,203853349030.0


q2_sales_decimal_sum: 240행 pandas와 정확히 일치


**해석:** 합계는 명목 원화이며 계절조정·물가조정 결과가 아닙니다. RDF 변환은 분석 단위와 입력 금액을 보존할 뿐 누락 매출을 추정하지 않습니다.

### 질의 3 — 공식/잠정 연결과 좌표 확보 상태

공식 확인 여부를 명시적으로 조회합니다. 좌표 확보는 시설 ID의 공간 활용 가능성을 뜻하며 공연·시설 연결의 정확성을 보증하지 않습니다.

In [18]:
q3 = run_query('q3_link_status', """
SELECT ?status_uri ?link_method ?coordinates (COUNT(DISTINCT ?performance) AS ?performance_ids)
WHERE {
  ?link a ex:FacilityLink; ex:linkedPerformance ?performance;
        ex:linkStatus ?status_uri; ex:linkMethod ?link_method.
  ?performance ex:coordinateAvailable ?coordinates.
} GROUP BY ?status_uri ?link_method ?coordinates ORDER BY ?status_uri ?link_method ?coordinates
""")
q3['status_uri'] = q3['status_uri'].map(str)
expected3 = performances.assign(status_uri=performances['status'].map(lambda s: str(EX[s]))).groupby(
    ['status_uri', 'link_method', 'coordinates']).agg(performance_ids=('mt20id', 'nunique')).reset_index()
compare('q3_link_status', q3, expected3, ['status_uri', 'link_method', 'coordinates'])

,status_uri,link_method,coordinates,performance_ids
0,https://example.org/grad-study/official,official_detail,True,899
1,https://example.org/grad-study/provisional,exact_unique_name,True,11940


q3_link_status: 2행 pandas와 정확히 일치


**해석:** 잠정 연결 공연을 공식 확인 공연과 합쳐 일반적인 확정 사실처럼 제시하지 않습니다. 후속 질의에서 `ex:linkStatus ex:official` 조건을 추가하면 공식 상세 ID로 확인된 연결만 선택할 수 있습니다.

### 질의 4 — 현재 상권 경계에서 500m 이내인 시설 관계

자치구는 시설 소재지가 아니라 **상권 소재지**입니다. 동일 시설이 여러 상권과 연결되므로 관계 수와 고유 시설 수를 구분합니다.

In [19]:
q4 = run_query('q4_proximity_500m', """
SELECT ?district (COUNT(?relation) AS ?relations) (COUNT(DISTINCT ?facility) AS ?facility_ids)
       (MIN(?distance) AS ?min_distance_m) (MAX(?distance) AS ?max_distance_m)
WHERE {
  ?relation a ex:ProximityRelation; ex:nearArea/ex:districtName ?district;
            ex:nearFacility ?facility; ex:distanceMetres ?distance; ex:distanceUnit ex:metre.
  FILTER(?distance <= 500)
} GROUP BY ?district ORDER BY ?district
""")
near_district = near.merge(areas[['area_id', 'district']], on='area_id', validate='many_to_one')
expected4 = near_district.groupby('district').agg(
    relations=('mt10id', 'size'), facility_ids=('mt10id', 'nunique'),
    min_distance_m=('distance_value', 'min'), max_distance_m=('distance_value', 'max')).reset_index()
compare('q4_distance_relations', q4, expected4, ['district'])

,district,relations,facility_ids,min_distance_m,max_distance_m
0,마포구,2822,347,0.0,499.9502197593884
1,종로구,2848,452,0.0,499.98505347129424


q4_distance_relations: 2행 pandas와 정확히 일치


**해석:** 거리는 과거 분기별 경계 변화나 보행 네트워크를 반영하지 않습니다. 0m는 현재 상권 폴리곤 내부에 있다는 뜻이며 실제 이동거리가 0이라는 뜻이 아닙니다.

### 질의 5 — 상권·분기별 주변 등록 공연 수 재현

시설 연결, 500m 거리, 활성분기를 연결하여 고유 공연 ID를 셉니다. SPARQL은 관계가 있는 조합만 반환합니다. 전체 상권×분기를 다시 맞춘 뒤 **확인된 관계의 건수**만 0으로 채워 03의 노출변수를 재현합니다. 매출에는 이 처리를 적용하지 않습니다.

In [20]:
q5 = run_query('q5_area_quarter_performances', """
SELECT ?area_id ?quarter (COUNT(DISTINCT ?performance) AS ?known_performance_count_500m)
WHERE {
  ?relation a ex:ProximityRelation; ex:nearArea ?area; ex:nearFacility ?facility;
            ex:distanceMetres ?distance.
  FILTER(?distance <= 500)
  ?area ex:areaCode ?area_id.
  ?link ex:linkedFacility ?facility; ex:linkedPerformance ?performance.
  ?performance ex:activeQuarter/ex:quarterCode ?quarter.
} GROUP BY ?area_id ?quarter ORDER BY ?area_id ?quarter
""")
area_quarters = panel[['area_id', 'quarter']].drop_duplicates()
column = 'known_performance_count_500m'
positive_counts = active.merge(near[['area_id', 'mt10id']], on='mt10id').groupby(
    ['area_id', 'quarter'])['mt20id'].nunique().rename(column).reset_index()
expected5 = area_quarters.merge(positive_counts, on=['area_id', 'quarter'], how='left')
full_q5 = area_quarters.merge(q5, on=['area_id', 'quarter'], how='left', validate='one_to_one')
for frame in [expected5, full_q5]:
    frame[column] = frame[column].fillna(0).astype(int)
compare('q5_graph_join_counts', full_q5, expected5, ['area_id', 'quarter'])
upstream5 = panel[['area_id', 'quarter', column]].drop_duplicates()
upstream5[column] = upstream5[column].astype(int)
compare('q5_matches_notebook08_panel', full_q5, upstream5, ['area_id', 'quarter'])
query_results['q5_area_quarter_performances'] = full_q5.sort_values(['area_id', 'quarter'])

,area_id,quarter,known_performance_count_500m
0,3001494,20231,130
1,3001494,20232,143
2,3001494,20233,175
3,3001494,20234,218
4,3001494,20241,135
5,3001494,20242,162
6,3001494,20243,170
7,3001494,20244,231


q5_graph_join_counts: 1,812행 pandas와 정확히 일치


,area_id,quarter,known_performance_count_500m
0,3001494,20231,130
1,3001494,20232,143
2,3001494,20233,175
3,3001494,20234,218
4,3001494,20241,135
5,3001494,20242,162
6,3001494,20243,170
7,3001494,20244,231


q5_matches_notebook08_panel: 1,812행 pandas와 정확히 일치


**해석:** 같은 상권·분기의 공연 노출은 외식업 10종에 공통입니다. 공연 수는 공간 관계의 고유 ID 수이고 실제 상연 회차 또는 방문객을 의미하지 않습니다. 0은 현재 확보한 시설·좌표·연결 자료에서 확인된 공연 ID가 없다는 뜻입니다. 잠정 연결과 수집 범위·시점의 한계가 있으므로 실제 무공연을 뜻하지 않습니다.

## 5. 무결성 검사와 재사용 파일 저장

관측별 금액 생략 규칙, 공식/잠정 구분, 좌표 누락 보존을 직접 검사하고 Turtle을 다시 읽어 같은 트리플이 보존되는지 확인합니다. 질의 5개의 실행 결과와 출처 해시·범위·검증 결과를 저장합니다. SHA-256은 입력 파일 변경 탐지용이며 자료가 정확하다는 보증은 아닙니다.

In [21]:
missing_nodes = {node('sales', f'{r.area_id}-{r.industry_id}-{r.quarter}')
                 for r in panel.loc[~panel['provided']].itertuples(index=False)}
assert all(not list(graph.objects(n, EX.totalSales)) for n in missing_nodes)
assert all(graph.value(n, EX.salesProvided).toPython() is False for n in missing_nodes)
assert len(set(graph.subjects(EX.totalSales))) == int(panel['provided'].sum())
assert all(len(list(graph.objects(node('sales', f'{r.area_id}-{r.industry_id}-{r.quarter}'),
                                    EX.totalSales))) == 1
           for r in panel.loc[panel['provided']].itertuples(index=False))
for row in facilities.loc[~facilities['coordinate_available']].itertuples(index=False):
    assert not list(graph.objects(node('facility', row.mt10id), EX.longitude))
    assert not list(graph.objects(node('facility', row.mt10id), EX.latitude))
assert len(set(graph.subjects(EX.linkStatus, EX.official))) == int(performances['official'].sum())
assert len(set(graph.subjects(EX.linkStatus, EX.provisional))) == int((~performances['official']).sum())
assert len(set(graph.subjects(RDF.type, EX.Industry))) == 10
assert all(hashlib.sha256((ROOT / row.path).read_bytes()).hexdigest() == row.sha256
           for row in source_manifest.itertuples(index=False))
validations.extend([
    {'check': 'all_ten_catalog_industries', 'status': 'passed', 'rows_checked': len(catalog)},
    {'check': 'source_files_unchanged', 'status': 'passed', 'rows_checked': len(source_manifest)},
    {'check': 'missing_sales_omit_value', 'status': 'passed', 'rows_checked': len(missing_nodes)},
    {'check': 'provided_sales_exactly_one_value', 'status': 'passed', 'rows_checked': int(panel['provided'].sum())},
    {'check': 'missing_coordinates_omit_values', 'status': 'passed',
     'rows_checked': int((~facilities['coordinate_available']).sum())},
    {'check': 'official_provisional_link_counts', 'status': 'passed', 'rows_checked': len(performances)},
])

In [22]:
vocab_path, graph_path = OUT / 'ontology.ttl', OUT / 'knowledge_graph.ttl'
vocab.serialize(destination=vocab_path, format='turtle', encoding='utf-8')
graph.serialize(destination=graph_path, format='turtle', encoding='utf-8')
roundtrip = Graph().parse(graph_path, format='turtle')
assert len(roundtrip) == len(graph)
assert all(triple in roundtrip for triple in graph)
assert len(Graph().parse(vocab_path, format='turtle')) == len(vocab)
validations.append({'check': 'turtle_roundtrip_exact_triples', 'status': 'passed', 'rows_checked': len(graph)})
del roundtrip
for name, table in query_results.items():
    table.to_csv(REPORTS / f'06_{name}.csv', index=False, encoding='utf-8-sig')
source_manifest.to_csv(REPORTS / '06_source_manifest.csv', index=False, encoding='utf-8-sig')
coverage.to_csv(REPORTS / '06_graph_coverage.csv', index=False, encoding='utf-8-sig')
entity_counts.to_csv(REPORTS / '06_entity_counts.csv', index=False, encoding='utf-8-sig')
validation_table = pd.DataFrame(validations)
validation_table.to_csv(REPORTS / '06_validation.csv', index=False, encoding='utf-8-sig')
display(validation_table)

,check,status,rows_checked
0,q1_coverage,passed,120
1,q2_sales_decimal_sum,passed,240
2,q3_link_status,passed,2
3,q4_distance_relations,passed,2
4,q5_graph_join_counts,passed,1812
5,q5_matches_notebook08_panel,passed,1812
6,all_ten_catalog_industries,passed,10
7,source_files_unchanged,passed,9
8,missing_sales_omit_value,passed,10459
9,provided_sales_exactly_one_value,passed,7661


In [23]:
summary = {
    'scope': {'districts': DISTRICTS, 'industry_ids': INDUSTRIES, 'quarters': QUARTERS,
              'industries': catalog[['industry_id', 'industry']].to_dict('records'),
              'panel_source': INPUTS['panel'],
              'facility_rule': 'study_district_facilities_union_any_study_area_500m_facilities',
              'performance_rule': 'performances_linked_to_included_facilities'},
    'seoul_refresh_run_id': scope_metadata.get('seoul_refresh_run_id'),
    'kopis_collection_run_id': scope_metadata['exposure_snapshot_linkage'].get('collection_run_id'),
    'triples': len(graph), 'vocabulary_triples': len(vocab),
    'areas': len(areas), 'facilities': len(facilities), 'performance_ids': len(performances),
    'seoul_performance_ids_in_input': len(frames['performances']),
    'excluded_performance_ids': len(frames['performances']) - len(performances),
    'input_performance_ids_without_facility': int(frames['performances']['mt10id'].isna().sum()),
    'official_links': int(performances['official'].sum()),
    'provisional_links': int((~performances['official']).sum()),
    'facilities_without_coordinates': int((~facilities['coordinate_available']).sum()),
    'performances_without_coordinates': int((~performances['coordinates']).sum()),
    'sales_observations': len(panel), 'sales_values': int(panel['provided'].sum()),
    'sales_values_omitted': int((~panel['provided']).sum()),
    'relations_500m': len(near), 'performance_quarter_relations': len(active),
    'sparql_queries': len(queries), 'pandas_comparisons': 6,
    'all_validations_passed': validation_table['status'].eq('passed').all().item(),
    'versions': {p: importlib.metadata.version(p) for p in ['pandas', 'rdflib']},
    'limitations': ['분기 단위 유지; 일별 인과 추론 불가', '등록 공연 ID 수는 상연 회차·관객수가 아님',
                    '시설명 연결은 잠정 연결', f'현재 좌표 미확인 시설 {int((~facilities["coordinate_available"]).sum())}개; 좌표 미확인 시 거리집계 제외',
                    '현재 상권 경계를 과거 분기에 공통 적용; 과거 시설 위치는 별도 검증하지 않음', '직접 입력 파일 수준 출처만 기록'],
    'outputs': [str(p.relative_to(ROOT)).replace('\\', '/') for p in [vocab_path, graph_path]],
}
(REPORTS / '06_knowledge_graph_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(Markdown(
    f"**검증 완료:** SPARQL {len(queries)}개, pandas 대조 6개와 "
    f"무결성·재읽기 검사를 통과했습니다. 데이터 {len(graph):,}개 트리플을 저장했습니다."))

**검증 완료:** SPARQL 5개, pandas 대조 6개와 무결성·재읽기 검사를 통과했습니다. 데이터 432,888개 트리플을 저장했습니다.

## 결과와 활용 범위

현재 RDF는 **총매출·공연노출 부분 그래프**이며 야간비중, 점포수, 09·10 노트북의 회귀 결과는 포함하지 않습니다.

이 그래프는 새로운 통계적 증거를 추가하지 않고 기존 자료의 개체·관계·단위·출처를 일관된 어휘로 드러냅니다. SPARQL 5개가 pandas와 일치하므로 기존 표 집계를 재현하는 작은 질의 계층으로 사용할 수 있습니다.

후속 질문 도구는 `data/analysis/knowledge_graph/`의 두 Turtle 파일을 로컬에서 읽고 이 노트북 코드 셀의 SPARQL 질의를 재사용할 수 있습니다. 답변에는 제공률, 공식/잠정 연결 구분, 분기 단위와 현재 경계·좌표 한계를 함께 표시해야 합니다. 이 노트북의 범위는 지식그래프와 검증 질의이며 질문 응답 서비스나 자동 에이전트 배포는 포함하지 않습니다.

- `data/analysis/knowledge_graph/ontology.ttl`: 클래스·속성 및 단위·집계규칙의 최소 어휘
- `data/analysis/knowledge_graph/knowledge_graph.ttl`: 범위를 제한한 개체·관측·관계
- `reports/06_*.csv`, `reports/06_knowledge_graph_summary.json`: 결과·출처 해시·검증 기록